In [1]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope
import mlflow
import numpy as np
import pandas as pd
from prefect import task, flow
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

/home/codespace/anaconda3/envs/mlflow_conda/lib/python3.13/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/codespace/anaconda3/envs/mlflow_conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("RandomState-hyperopt")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1773843100059, experiment_id='1', last_update_time=1773843100059, lifecycle_stage='active', name='RandomState-hyperopt', tags={}, workspace='default'>

In [3]:
file_path = './data/yellow_tripdata_2023-03.parquet'
def load(file: str) -> pd.DataFrame:
    df = pd.read_parquet(file)
    df['duration'] = df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    # df.head(5)
    return df

In [ ]:
df = load(file_path)

In [4]:
def preprocess(df: pd.DataFrame, fit_dv: bool=True) -> tuple:
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    df = df.head(500)
    df_encoded = pd.get_dummies(df[['PU_DO', 'trip_distance']], columns=['PU_DO'])
    return df_encoded

In [21]:
y_train = df['duration'].head(500).values

In [6]:
X = preprocess(df)

In [25]:
X_train, X_val, Y_train, Y_val = train_test_split(X, y_train, test_size=0.2, random_state=42)

In [23]:
# search_space = {
#     "max_depth": scope.int(hp.quniform('max_depth', 4, 100, 1)),
#     "learning_rate": hp.loguniform('learning_rate', -3, 0), # exp(-3) - exp(0) -> [0.05, 1]
#     # "reg_alpha": hp.loguniform('reg_alpha', -5, -1),
#     # "reg_lambda": hp.loguniform('reg_lambda', -6, -1),
#     "min_child_weight": hp.loguniform('min_child_weight', -1, 3),
#     "seed": 42
# }

search_space = {
        'max_depth': scope.int(hp.quniform('max_depth', 1, 20, 1)),
        'n_estimators': scope.int(hp.quniform('n_estimators', 10, 50, 1)),
        'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1)),
        'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 4, 1)),
        'random_state': 42
    }

def train(x: pd.DataFrame, y: np.ndarray, x_val: pd.DataFrame, y_val: np.ndarray, params: dict):
    with mlflow.start_run():
        mlflow.set_tag("author", "ahmed")
        mlflow.sklearn.autolog()
        
        model = RandomForestRegressor(**params)
        model.fit(x, y)
        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_pred, y_val)

        mlflow.log_metrics({"rmse": rmse})

    return {"rmse": rmse, "status": STATUS_OK}

rstate = np.random.defualt_rng(42)

fmin(
    fn=train,
    algo=tpe.suggets,
    max_eval=10,
    trials=Trials,
    rstate=rstate
)

NameError: name 'numpy' is not defined